In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import random
import os
from collections import deque

np.random.seed(0)
random.seed(0)
torch.manual_seed(0)
os.environ["PYTHONHASHSEED"] = str(0)

# --- CONFIGURATION ---
N = 10  # Grid Size
S = N * N
RANK = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
# 1. DATASET UTILITIES

def get_grid_target(idx, action):
    row, col = idx // N, idx % N
    if action == 0 and row > 0: row -= 1    # Up
    elif action == 1 and row < N-1: row += 1 # Down
    elif action == 2 and col > 0: col -= 1   # Left
    elif action == 3 and col < N-1: col += 1 # Right
    return row * N + col

def bfs_check_path(obs_mask, start, goal):
    if obs_mask[start] == 1.0 or obs_mask[goal] == 1.0: return False
    q = deque([[start,0]])
    visited = {start}
    while q:
        curr, d = q.popleft()
        if curr == goal: return True, d
        row, col = curr // N, curr % N
        for dr, dc in [(0,1), (0,-1), (1,0), (-1,0)]:
            nr, nc = row + dr, col + dc
            if 0 <= nr < N and 0 <= nc < N:
                nxt = nr * N + nc
                if obs_mask[nxt] == 0.0 and nxt not in visited:
                    visited.add(nxt)
                    q.append([nxt,d+1])
    return False, 0

def create_batch_with_goals(batch_size=64, empty_grid=False):
    """
    Returns (obs, cur, target, acts, goals)
    We need 'goals' now to train the Policy Head.
    """
    obs_masks, cur_states, target_moves, actions, goals = [], [], [], [], []
    for _ in range(batch_size):
        mask = torch.zeros(S)
        if not empty_grid:
            obs_indices = random.sample(range(S), int(S * 0.20))
            mask[obs_indices] = 1.0

        start = random.randint(0, S-1)
        # Goal must be valid and distinct
        goal = random.randint(0, S-1)
        while mask[goal] == 1.0 or goal == start:
            goal = random.randint(0, S-1)

        # Random action for Dynamics Training
        action = random.randint(0, 3)
        target = get_grid_target(start, action)
        mask[start], mask[target] = 0.0, 0.0

        obs_masks.append(mask)
        cur_states.append(start)
        target_moves.append(target)
        actions.append(action)
        goals.append(goal)

    return (torch.stack(obs_masks).to(DEVICE),
            torch.tensor(cur_states).to(DEVICE),
            torch.tensor(target_moves).to(DEVICE),
            torch.tensor(actions).to(DEVICE),
            torch.tensor(goals).to(DEVICE))

def create_solvable_batch(batch_size=1):
    obs_masks, cur_states, goals, step_count = [], [], [], 0
    while len(obs_masks) < batch_size:
        mask = torch.zeros(S)
        obs_indices = random.sample(range(S), int(S * 0.20))
        mask[obs_indices] = 1.0
        start = random.randint(0, S-1)
        goal = random.randint(0, S-1)
        if start != goal and mask[start] == 0 and mask[goal] == 0:
            flag, count = bfs_check_path(mask.numpy(), start, goal)
            if flag:
                obs_masks.append(mask)
                cur_states.append(start)
                goals.append(goal)
                step_count += count
    return (torch.stack(obs_masks).to(DEVICE),
            torch.tensor(cur_states).to(DEVICE),
            torch.tensor(goals).to(DEVICE)
             ,step_count)

In [3]:
class MazeSolver(nn.Module):
    def __init__(self, N, rank=32):
        super().__init__()
        self.N = N; self.S = N * N

        # Motor: Operator Calculus (Dynamics)
        self.L = nn.Parameter(torch.randn(4, self.S, rank) * 0.1)
        self.R = nn.Parameter(torch.randn(4, self.S, rank) * 0.1)

        # Energy-Seeker: Physics (Map)
        self.V_head = nn.Sequential(nn.Linear(self.S, 128), nn.ReLU(), nn.Linear(128, self.S))
        self.m_head = nn.Sequential(nn.Linear(self.S, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid())

        # --- NEW: SYSTEM 1 (FAST) POLICY HEAD ---
        # Input: [OneHot(State) + OneHot(Goal)] -> 2*S features
        # Output: 4 Logits (Up, Down, Left, Right)
        self.policy_head = nn.Sequential(
            nn.Linear(2 * self.S, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 4)
        )

        self.register_buffer('K_grid', self._build_laplacian(N))

    def _build_laplacian(self, n):
        K = torch.zeros(self.S, self.S)
        for i in range(self.S):
            x, y = i % n, i // n
            for dx, dy in [(0,1),(0,-1),(1,0),(-1,0)]:
                nx, ny = x+dx, y+dy
                if 0 <= nx < n and 0 <= ny < n:
                    j = ny * n + nx
                    K[i, i] += 1.0; K[i, j] = -1.0
        return K

    def get_hamiltonian(self, obs_mask):
        """Constructs H for Training."""
        m = 0.1 + 9.9 * self.m_head(1.0 - obs_mask)
        m_inv = 1.0 / (m.unsqueeze(-1) + 1e-6)
        V_env = obs_mask * 100.0
        V_internal = self.V_head(1.0 - obs_mask)
        V_total = V_internal + V_env
        K_batch = self.K_grid.unsqueeze(0).expand(obs_mask.size(0), -1, -1)
        H = m_inv * K_batch + torch.diag_embed(V_total)
        H = (H + H.transpose(-2, -1)) * 0.5
        jitter = torch.eye(self.S, device=H.device).unsqueeze(0) * 1e-3
        H = H + jitter
        return H

    def get_hamiltonian_for_navigation(self, obs_mask, goal_idx):
        """Constructs H for Inference (Superfluid Regime)."""
        m_inv = 1.0 / 0.05
        V_total = obs_mask * 100.0

        # Handle goal_idx being either tensor of indices or one-hot
        if goal_idx.dim() == 1:
            goal_mask = F.one_hot(goal_idx, num_classes=self.S).float()
        else:
            goal_mask = goal_idx # Already one-hot or features

        V_total = V_total - (goal_mask * 100.0)

        K_batch = self.K_grid.unsqueeze(0).expand(obs_mask.size(0), -1, -1)
        H = m_inv * K_batch + torch.diag_embed(V_total)
        H = (H + H.transpose(-2, -1)) * 0.5
        jitter = torch.eye(self.S, device=H.device).unsqueeze(0) * 1e-3
        H = H + jitter
        return H

    def get_op(self, act_idx):
        return self.L[act_idx] @ self.R[act_idx].transpose(-2, -1)

    def forward(self, cur_idx, act_idx):
        psi = F.one_hot(cur_idx, num_classes=self.S).float()
        ops = torch.stack([self.get_op(a) for a in act_idx])
        preds = torch.bmm(psi.unsqueeze(1), ops).squeeze(1)
        return preds

    def get_policy(self, cur_idx, goal_idx):
        cur_feat = F.one_hot(cur_idx, num_classes=self.S).float()
        if isinstance(goal_idx, int):
            goal_idx = torch.tensor([goal_idx] * cur_idx.size(0)).to(cur_idx.device)
        if goal_idx.dim() == 1:
            goal_feat = F.one_hot(goal_idx, num_classes=self.S).float()
        else:
            goal_feat = goal_idx

        x = torch.cat([cur_feat, goal_feat], dim=1)
        return self.policy_head(x)


In [4]:
# 3 stage training
def gated_training_loop(agent):
    optimizer = optim.Adam(agent.parameters(), lr=0.002)
    batch_size = 64

    # --- STAGE 1: MOTOR SKILLS (Standard) ---
    print("\n>>> STAGE 1: MOTOR PRE-TRAINING (Target: >98% Acc)...")
    agent.L.requires_grad = True; agent.R.requires_grad = True
    for p in agent.V_head.parameters(): p.requires_grad = False
    for p in agent.policy_head.parameters(): p.requires_grad = False

    epoch = 0; motor_acc = 0.0
    while motor_acc < 0.98:
        epoch += 1
        obs, cur, target, acts, _ = create_batch_with_goals(batch_size, empty_grid=True)
        optimizer.zero_grad()
        preds = agent(cur, acts)
        loss = F.cross_entropy(preds, target)
        loss.backward(); optimizer.step()
        motor_acc = (preds.argmax(dim=1) == target).float().mean().item()

        if epoch % 20 == 0:
            print(f"Stage 1 | Epoch {epoch} | Loss: {loss.item():.4f} | Motor Acc: {motor_acc:.2%}")
            if epoch > 600:
                for pg in optimizer.param_groups: pg['lr'] *= 0.8
    print(">>> MOTOR SKILLS ACQUIRED.")

    # --- STAGE 2: PHYSICS ANCHORING (Standard) ---
    print("\n>>> STAGE 2: ANCHORING PHYSICS (Target: Zero Wall Overlap)...")
    agent.L.requires_grad = False; agent.R.requires_grad = False
    for p in agent.V_head.parameters(): p.requires_grad = True
    for p in agent.policy_head.parameters(): p.requires_grad = False

    for epoch in range(1, 41):
        obs, cur, target, acts, _ = create_batch_with_goals(batch_size, empty_grid=False)
        optimizer.zero_grad()
        H = agent.get_hamiltonian(obs)
        try:
            vals, vecs = torch.linalg.eigh(H.to(torch.float64))
            psi0_sq = vecs[:, :, 0].pow(2).to(torch.float32)
            loss = torch.mean(torch.sum(psi0_sq * obs, dim=1)) * 100.0
            loss.backward(); optimizer.step()
            if epoch % 10 == 0:
                print(f"Stage 2 | Epoch {epoch} | Wall Overlap Loss: {loss.item():.6f}")
        except torch._C._LinAlgError:
            optimizer.zero_grad()

    # --- STAGE 3: UNIFIED INTEGRATION & HARD DISTILLATION ---
    print("\n>>> STAGE 3: UNIFIED INTEGRATION & HARD POLICY DISTILLATION...")
    agent.L.requires_grad = True; agent.R.requires_grad = True
    for p in agent.V_head.parameters(): p.requires_grad = False
    for p in agent.policy_head.parameters(): p.requires_grad = True

    for epoch in range(1, 101):
        obs, cur, target, acts, goals = create_batch_with_goals(batch_size, empty_grid=False)
        optimizer.zero_grad()

        # 1. Dynamics Loss
        preds = agent(cur, acts)
        loss_dynamics = F.cross_entropy(preds, target)

        # 2. Physics Constraint
        with torch.no_grad(): H = agent.get_hamiltonian(obs)
        psi_next = F.softmax(preds, dim=1)
        energy_pen = torch.mean(torch.sum(psi_next * (psi_next @ H), dim=1))

        # 3. Policy Distillation (HARD TARGETS)
        with torch.no_grad():
            H_nav = agent.get_hamiltonian_for_navigation(obs, goals)
            vals, vecs = torch.linalg.eigh(H_nav.to(torch.float64))
            psi_flow = vecs[:, :, 0].pow(2).to(torch.float32) # [B, S]

            # Construct HARD Targets (Index of the best neighbor)
            target_indices = []
            for b in range(batch_size):
                c = cur[b].item()
                r, col = c // N, c % N

                # Check neighbors: Up(0), Down(1), Left(2), Right(3)
                coords = [(0, r-1, col), (1, r+1, col), (2, r, col-1), (3, r, col+1)]

                best_act = -1
                max_flux = -1.0

                for act, nr, nc in coords:
                    if 0<=nr<N and 0<=nc<N:
                        node = nr*N + nc
                        # Now this index works because psi_flow is [B, S]
                        flux = psi_flow[b, node].item()
                        if flux > max_flux:
                            max_flux = flux
                            best_act = act
                    else:
                        pass # Wall/OOB

                # If stuck (rare), random valid
                if best_act == -1: best_act = random.randint(0, 3)
                target_indices.append(best_act)

            target_indices = torch.tensor(target_indices).to(DEVICE)

        # Train Policy Head with CrossEntropy (Sharp Classification)
        policy_logits = agent.get_policy(cur, goals)
        loss_policy = F.cross_entropy(policy_logits, target_indices)

        # Total Loss
        loss = loss_dynamics + (0.05 * energy_pen) + (5.0 * loss_policy)

        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            print(f"Stage 3 | Epoch {epoch} | Dyn: {loss_dynamics:.3f} | Pol: {loss_policy:.3f}")


In [5]:
#visualize the output model plan
import matplotlib.pyplot as plt
import os

def visualize_dual_system_run(run_id, N, obs_mask, start, goal, path_history, flux_map=None, outdir="viz_results"):
    """
    Visualizes a single navigation run with Dual-System annotations.

    Args:
        run_id (int): Identifier for the file name.
        N (int): Grid size.
        obs_mask (Tensor): 1D tensor of wall locations (1.0 = wall).
        start (int): Start node index.
        goal (int): Goal node index.
        path_history (list): List of tuples (current_node, next_node, system_label).
                             system_label should be 'sys1' or 'sys2'.
        flux_map (Tensor/Array): Optional 1D array of flux values for the right panel.
        outdir (str): Directory to save images.
    """
    os.makedirs(outdir, exist_ok=True)

    # 1. Setup Grid
    grid = obs_mask.cpu().numpy().reshape(N, N)

    # Create Figure
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # --- PANEL 1: THE TRAJECTORY (Behavior) ---
    ax = axes[0]
    ax.imshow(grid, cmap='Greys', origin='upper', vmin=0, vmax=1)

    # Plot Path Segments
    for (u, v, sys_type) in path_history:
        uy, ux = divmod(u, N)
        vy, vx = divmod(v, N)

        color = 'blue' if sys_type == 'sys1' else 'red'
        alpha = 0.6 if sys_type == 'sys1' else 1.0
        width = 1.5 if sys_type == 'sys1' else 2.5

        ax.plot([ux, vx], [uy, vy], color=color, linewidth=width, alpha=alpha)
        # Add slight dot at vertices to smooth look
        ax.plot([ux], [uy], '.', color=color, markersize=5)

    # Mark Start/Goal
    sy, sx = divmod(start, N)
    gy, gx = divmod(goal, N)
    ax.scatter([sx], [sy], c='lime', s=150, edgecolors='black', label='Start', zorder=10)
    ax.scatter([gx], [gy], c='gold', marker='*', s=200, edgecolors='black', label='Goal', zorder=10)

    # Legend for Systems
    from matplotlib.lines import Line2D
    custom_lines = [Line2D([0], [0], color='blue', lw=2),
                    Line2D([0], [0], color='red', lw=2)]
    ax.legend(custom_lines, ['System 1 (Habit)', 'System 2 (Physics)'], loc='lower right', fontsize='small')
    ax.set_title(f"Run {run_id}: Dual-System Trajectory")
    ax.axis('off')

    # --- PANEL 2: THE TRIGGERS (Cognition) ---
    ax = axes[1]
    ax.imshow(grid, cmap='Greys', origin='upper')
    ax.set_title("System 2 Activation Sites")

    # Plot only System 2 activations
    sys2_points = [u for (u, v, sys_type) in path_history if sys_type == 'sys2']
    if sys2_points:
        ys, xs = [], []
        for p in sys2_points:
            r, c = divmod(p, N)
            ys.append(r); xs.append(c)
        ax.scatter(xs, ys, c='red', s=80, alpha=0.7, edgecolors='white', label='Re-planning Event')
        ax.legend(loc='lower right')
    else:
        ax.text(N//2, N//2, "System 1 Only", ha='center', color='blue')

    # Mark Goal ghost for reference
    ax.scatter([gx], [gy], c='gold', marker='*', s=100, edgecolors='grey', alpha=0.5)
    ax.axis('off')

    # --- PANEL 3: THE COMPASS (Physics) ---
    ax = axes[2]
    if flux_map is not None:
        if hasattr(flux_map, 'cpu'): flux_map = flux_map.cpu().numpy()
        flux_img = flux_map.reshape(N, N)
        im = ax.imshow(flux_img, cmap='viridis', origin='upper')
        ax.set_title("System 2: Superfluid Flux ($\\psi^2$)")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    else:
        ax.imshow(grid, cmap='Greys')
        ax.text(N//2, N//2, "No Flux Data", ha='center')
        ax.set_title("Superfluid Flux")

    ax.scatter([gx], [gy], c='red', marker='x', label='Goal')
    ax.axis('off')

    plt.tight_layout()
    save_path = os.path.join(outdir, f"dual_system_run_{run_id}.png")
    plt.savefig(save_path, dpi=500, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved visualization to {save_path}")

def dual_system_entropy_test(model, n_mazes=2000, entropy_threshold=0.6):
    model.eval()
    successes = 0; collisions = 0; timeouts = 0
    sys2_triggers = 0; total_steps = 0

    # Visualization counters
    saved_viz_count = 0
    MAX_VIZ = 20

    print(f"\n>>> STARTING DUAL-SYSTEM TEST (Threshold={entropy_threshold} + MEMORY + PRUNING)...")
    average = 0
    average_bfs = 0
    for i in range(n_mazes):
        obs, start, goals, count = create_solvable_batch(1)
        average_bfs += count
        goal_idx = goals[0].item()
        cur = start.item()

        psi_flow_cache = None
        wall_map = obs.squeeze(0).clone() # Clone so we can modify it (Pruning)
        visits = torch.zeros(model.S).to(DEVICE)
        path_history = []

        for step in range(200):
            if cur == goal_idx: successes += 1; break
            visits[cur] += 1.0
            total_steps += 1

            prev_node = cur

            # --- DYNAMIC PRUNING ---
            # If we are in a dead-end (only 1 valid neighbor), mark it as a wall
            # so we treat it as "closed" once we move.
            # This does not persist across episodes and does not use future information.

            r0, c0 = cur // model.N, cur % model.N
            neighbors = []
            for dr, dc in [(-1,0), (1,0), (0,-1), (0,1)]:
                nr, nc = r0+dr, c0+dc
                if 0<=nr<model.N and 0<=nc<model.N:
                    n_idx = nr*model.N + nc
                    if wall_map[n_idx] == 0.0:
                        neighbors.append(n_idx)

            # If 1 exit (and not start/goal), it's a dead end tip.
            # We mark it as wall effectively "zipping" the dead end behind us.
            if len(neighbors) <= 1 and cur != start.item() and cur != goal_idx:
                wall_map[cur] = 1.0

            # --- STEP A: SYSTEM 1 PREDICTION ---
            with torch.no_grad():
                s1_logits = model.get_policy(torch.tensor([cur]).to(DEVICE),
                                           torch.tensor([goal_idx]).to(DEVICE))
                s1_probs = F.softmax(s1_logits, dim=1)

            current_entropy = -torch.sum(s1_probs * torch.log(s1_probs + 1e-8)).item()
            best_s1_act = s1_probs.argmax().item()

            # Compute Target for S1
            r, c = cur // model.N, cur % model.N
            if best_s1_act==0: r-=1
            elif best_s1_act==1: r+=1
            elif best_s1_act==2: c-=1
            elif best_s1_act==3: c+=1

            if 0<=r<model.N and 0<=c<model.N: best_s1_node = r*model.N + c
            else: best_s1_node = cur

            # --- STEP B: THE SWITCH ---
            is_confused = (current_entropy > entropy_threshold)

            # Safety Check (Uses updated wall_map, so S1 won't re-enter pruned dead-ends)
            is_unsafe = (wall_map[best_s1_node] == 1.0) or (best_s1_node == cur)

            is_repetitive = (visits[best_s1_node] > 0)

            system_label = 'sys1'

            if not (is_confused or is_unsafe or is_repetitive):
                # >>> SYSTEM 1
                cur = best_s1_node
            else:
                # >>> SYSTEM 2: UNIFIED REASONING WITH INVALID MOVE FILTERING
                system_label = 'sys2'
                sys2_triggers += 1

                if psi_flow_cache is None:
                    with torch.no_grad():
                        # Lower mass (m) allows wave to tunnel/spread through barriers
                        # providing a gradient even far from the goal.
                        H = model.get_hamiltonian_for_navigation(obs, goals)
                        vals, vecs = torch.linalg.eigh(H.to(torch.float64))
                        # Use the ground state to find the 'global path'[cite: 145].
                        psi_flow_cache = vecs[0, :, 0].pow(2).to(torch.float32)

                best_node = None
                max_score = -1e9

                # Check all 4 symbolic operators: Up, Down, Left, Right [cite: 188]
                # Inside the System 2 (else) block:
                for act in range(4):
                    with torch.no_grad():
                        pred_wave = model(torch.tensor([cur]).to(DEVICE), torch.tensor([act]).to(DEVICE))
                        pred_dist = F.softmax(pred_wave, dim=1).squeeze(0)

                    predicted_destination = pred_dist.argmax().item()

                    # --- ADD THIS STRICT ADJACENCY CHECK ---
                    r0, c0 = divmod(cur, model.N)
                    r1, c1 = divmod(predicted_destination, model.N)
                    is_adjacent = (abs(r0 - r1) + abs(c0 - c1)) == 1

                    if not is_adjacent: continue # Veto "teleportation"
                    if wall_map[predicted_destination] == 1.0: continue # Veto walls

                    # 2. Check if the action actually moves the agent (prevents walking into edges).
                    if predicted_destination == cur:
                        continue

                    # Pillar 2: Score the valid move based on its alignment with the goal-flux[cite: 161].
                    expected_flux = torch.sum(pred_dist * psi_flow_cache).item()

                    # Apply memory penalty to prevent backtracking loops.
                    penalty = 1.0 / (1.0 + visits[predicted_destination] * 10.0)
                    score = expected_flux * penalty

                    if score > max_score:
                        max_score = score
                        best_node = predicted_destination

                if best_node is None:
                    break # All possible actions are blocked or invalid

                cur = best_node

            path_history.append((prev_node, cur, system_label))
            if wall_map[cur] == 1.0 and cur != goal_idx: collisions += 1; break
        average += len(path_history)
        if cur != goal_idx and wall_map[cur] == 0.0:
            timeouts += 1

        # --- VISUALIZATION ---
        if saved_viz_count < MAX_VIZ:
            flux_for_plot = None
            if psi_flow_cache is not None: flux_for_plot = psi_flow_cache
            else:
                with torch.no_grad():
                    H_viz = model.get_hamiltonian_for_navigation(obs, goals)
                    vals, vecs = torch.linalg.eigh(H_viz.to(torch.float64))
                    flux_for_plot = vecs[0, :, 0].pow(2)

            visualize_dual_system_run(
                run_id=i, N=model.N, obs_mask=obs[0], start=start.item(),
                goal=goal_idx, path_history=path_history, flux_map=flux_for_plot
            )
            saved_viz_count += 1
    print(f"Average step taken: {average/n_mazes:.2f} steps; BFS Average Step: {average_bfs/n_mazes:.2f}")
    print(f"Goal Success Rate: {(successes/n_mazes)*100:.1f}%")
    print(f"Wall Collision Rate: {(collisions/n_mazes)*100:.1f}%")
    print(f"System 2 Activation Rate: {(sys2_triggers/total_steps)*100:.1f}%")


In [6]:
agent = MazeSolver(N).to(DEVICE)
params = 0
for name, p in agent.named_parameters():
    params += p.numel()
print(f"There are {params} parameters in total")
gated_training_loop(agent)
torch.save(agent.state_dict(), "maze_solver_dual.pth")
# agent.load_state_dict(torch.load("maze_solver_dual.pth"))
dual_system_entropy_test(agent, 200, entropy_threshold=5.0)

There are 142825 parameters in total

>>> STAGE 1: MOTOR PRE-TRAINING (Target: >98% Acc)...
Stage 1 | Epoch 20 | Loss: 4.5510 | Motor Acc: 6.25%
Stage 1 | Epoch 40 | Loss: 4.4700 | Motor Acc: 45.31%
Stage 1 | Epoch 60 | Loss: 4.3836 | Motor Acc: 70.31%
Stage 1 | Epoch 80 | Loss: 4.3011 | Motor Acc: 90.62%
>>> MOTOR SKILLS ACQUIRED.

>>> STAGE 2: ANCHORING PHYSICS (Target: Zero Wall Overlap)...
Stage 2 | Epoch 10 | Wall Overlap Loss: 0.000079
Stage 2 | Epoch 20 | Wall Overlap Loss: 0.000073
Stage 2 | Epoch 30 | Wall Overlap Loss: 0.000074
Stage 2 | Epoch 40 | Wall Overlap Loss: 0.000052

>>> STAGE 3: UNIFIED INTEGRATION & HARD POLICY DISTILLATION...
Stage 3 | Epoch 10 | Dyn: 4.150 | Pol: 1.373
Stage 3 | Epoch 20 | Dyn: 4.100 | Pol: 1.343
Stage 3 | Epoch 30 | Dyn: 4.037 | Pol: 1.209
Stage 3 | Epoch 40 | Dyn: 3.962 | Pol: 0.870
Stage 3 | Epoch 50 | Dyn: 3.880 | Pol: 0.717
Stage 3 | Epoch 60 | Dyn: 3.773 | Pol: 0.502
Stage 3 | Epoch 70 | Dyn: 3.612 | Pol: 0.401
Stage 3 | Epoch 80 | Dyn: 3.

## See the images of maze navigation in the created viz_results folder